In [36]:
import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

In [37]:
def make_cv_folds(train_pool, dates, n_splits=5):
    for fold, (train_idx, val_idx) in enumerate(TimeSeriesSplit(n_splits=n_splits).split(dates)):
        train_dates = dates[train_idx]# gives the training dates for that specific fold like what the actual dates are
        val_dates = dates[val_idx]# gives the validation dates for that specific fold like what the actual validation dates are
        assert not set(train_dates) & set(val_dates)
        train_mask = train_pool['FlightDate'].isin(train_dates)# finds the rows that correspond to the dates and returns a boolean array
        val_mask = train_pool['FlightDate'].isin(val_dates)# finds the rows that correspond to the dates and returns a boolean array
        train_fold = train_pool[train_mask]#gets the actual flights rows that belong to the training dates
        val_fold = train_pool[val_mask]# gets the actual flight rows that belong to the validation dates
        yield fold, train_fold, val_fold, train_dates, val_dates

In [38]:
df = pd.read_csv('../data/interim/seattle_ontime_clean.csv') #Loading the csv
df.shape
df.columns


Index(['Unnamed: 0', 'Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek',
       'FlightDate', 'Reporting_Airline', 'DOT_ID_Reporting_Airline',
       'IATA_CODE_Reporting_Airline',
       ...
       'Div4TailNum', 'Div5Airport', 'Div5AirportID', 'Div5AirportSeqID',
       'Div5WheelsOn', 'Div5TotalGTime', 'Div5LongestGTime', 'Div5WheelsOff',
       'Div5TailNum', 'Unnamed: 109'],
      dtype='object', length=111)

In [39]:

df['FlightDate'] = pd.to_datetime(df["FlightDate"])
df['FlightDate'].head()

0   2024-08-01
1   2024-08-02
2   2024-08-03
3   2024-08-04
4   2024-08-01
Name: FlightDate, dtype: datetime64[ns]

In [40]:
post_flight = [
    'DepTime', 'DepDelay', 'DepDelayMinutes', 'DepDel15', 'DepartureDelayGroups',
    'TaxiOut', 'WheelsOff', 'WheelsOn', 'TaxiIn', 'ArrTime', 'ArrDelayMinutes',
    'ArrDel15', 'ArrivalDelayGroups', 'ActualElapsedTime', 'AirTime',
    'CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay',
    'FirstDepTime', 'TotalAddGTime', 'LongestAddGTime',
    'Cancelled', 'CancellationCode', 'Diverted',
    'DivAirportLandings', 'DivReachedDest', 'DivActualElapsedTime', 'DivArrDelay',
    'DivDistance',
    'Div1Airport', 'Div1AirportID', 'Div1AirportSeqID', 'Div1WheelsOn',
    'Div1TotalGTime', 'Div1LongestGTime', 'Div1WheelsOff', 'Div1TailNum',
    'Div2Airport', 'Div2AirportID', 'Div2AirportSeqID', 'Div2WheelsOn',
    'Div2TotalGTime', 'Div2LongestGTime', 'Div2WheelsOff', 'Div2TailNum',
    'Div3Airport', 'Div3AirportID', 'Div3AirportSeqID', 'Div3WheelsOn',
    'Div3TotalGTime', 'Div3LongestGTime', 'Div3WheelsOff', 'Div3TailNum',
    'Div4Airport', 'Div4AirportID', 'Div4AirportSeqID', 'Div4WheelsOn',
    'Div4TotalGTime', 'Div4LongestGTime', 'Div4WheelsOff', 'Div4TailNum',
    'Div5Airport', 'Div5AirportID', 'Div5AirportSeqID', 'Div5WheelsOn',
    'Div5TotalGTime', 'Div5LongestGTime', 'Div5WheelsOff', 'Div5TailNum',
]
drop_cols = [
    'Reporting_Airline', 'DOT_ID_Reporting_Airline', 'Tail_Number',
    'OriginAirportID', 'OriginAirportSeqID', 'OriginCityMarketID',
    'OriginCityName', 'OriginState', 'OriginStateFips', 'OriginStateName', 'OriginWac',
    'DestAirportID', 'DestAirportSeqID', 'DestCityMarketID',
    'DestCityName', 'DestState', 'DestStateFips', 'DestStateName', 'DestWac',
    'DepTimeBlk', 'ArrTimeBlk', 'Flights', 'DistanceGroup','Origin','Unnamed: 0', 'Unnamed: 109'
]


df = df.drop(columns=post_flight + drop_cols)
df.columns

Index(['Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'FlightDate',
       'IATA_CODE_Reporting_Airline', 'Flight_Number_Reporting_Airline',
       'Dest', 'CRSDepTime', 'CRSArrTime', 'ArrDelay', 'CRSElapsedTime',
       'Distance'],
      dtype='object')

## Why post-flight and redundant columns were dropped

Post-flight features like `DepDelay`, `DepTime`, `TaxiOut`, and `ActualElapsedTime` were dropped because they don't exist when someone is booking a flight. If we trained the model on these, it would pick up on things like "flights that depart late tend to arrive late" and look really accurate but that's useless because we wouldn't know the actual departure time when a user is trying to decide between two flights. We'd be feeding the model information it would never have in the real scenario and therefore it's predictions will be wrong.

The columns in `drop_cols` like `Reporting_Airline`, `OriginCityName`, `DistanceGroup`, and `DepTimeBlk` are just duplicates of information we already have in other columns. `Origin` was also dropped since every single row is SEA and therefore there's not really a pattern for the model to uncover with regards to predicting ArrDelay as both a delayed flight and an ontime flight have origin as SEA.

In [41]:
cutoff = pd.Timestamp('2025-10-01')#this is the cutoff because it correlates exactly with Q4 and also because it leaves plenty of data for training like about 88% for training and 12% for testing so that theres enough data to train on.
df['DepHour'] = df['CRSDepTime'] // 100
train_pool = df[df['FlightDate'] < cutoff] #gets all the rows to train on that are before this particular date and it also happens to be about 88% of the data 
test = df[df['FlightDate'] >= cutoff] # gets all the rows to test on that are after or on this date which also happens to be about 12% of the data

dates = np.sort(train_pool['FlightDate'].unique())# sorts the dates from training pool to unique dates basically gives you all the unique dates
for fold, train_fold, val_fold, train_dates, val_dates in make_cv_folds(train_pool, dates):
    print(fold, train_fold.shape[0], val_fold.shape[0], train_dates.min(), train_dates.max(), val_dates.min(), val_dates.max())


print(len(train_pool), train_pool['FlightDate'].nunique(), train_pool['FlightDate'].min(), train_pool['FlightDate'].max())
print(len(test), test['FlightDate'].nunique(), test['FlightDate'].min(), test['FlightDate'].max())
assert train_pool['FlightDate'].max() < test['FlightDate'].min()
assert len(train_pool) + len(test) == len(df)


0 41901 51554 2024-01-01T00:00:00.000000000 2024-04-18T00:00:00.000000000 2024-04-19T00:00:00.000000000 2024-08-02T00:00:00.000000000
1 93455 49684 2024-01-01T00:00:00.000000000 2024-08-02T00:00:00.000000000 2024-08-03T00:00:00.000000000 2024-11-16T00:00:00.000000000
2 143139 41431 2024-01-01T00:00:00.000000000 2024-11-16T00:00:00.000000000 2024-11-17T00:00:00.000000000 2025-03-02T00:00:00.000000000
3 184570 47184 2024-01-01T00:00:00.000000000 2025-03-02T00:00:00.000000000 2025-03-03T00:00:00.000000000 2025-06-16T00:00:00.000000000
4 231754 53895 2024-01-01T00:00:00.000000000 2025-06-16T00:00:00.000000000 2025-06-17T00:00:00.000000000 2025-09-30T00:00:00.000000000
285649 639 2024-01-01 00:00:00 2025-09-30 00:00:00
38841 92 2025-10-01 00:00:00 2025-12-31 00:00:00


## Train pool / test split and CV strategy (Issue 2)

Single cutoff date: `2025-10-01`. Everything before it is the **train pool**
(2024-01-01 to 2025-09-30, 639 dates, 285,649 rows); everything on or after it is
**test** (2025-10-01 to 2025-12-31, 92 dates, 38,841 rows, ~12% of the data).
Test is held out untouched until Issue 10 — it is never used to compare or select
models.

There is no separate fixed validation split. Every candidate model in Issues 3, 4,
5, 8, 9 is scored across the same 5 `TimeSeriesSplit` folds over the train pool, so
every comparison uses identical folds.

`TimeSeriesSplit` is run over the array of the train pool's *unique dates*, not over
rows, because flight rows are not equally spaced (242-555 flights/day) while calendar
dates are. Each fold's row membership is recovered afterward with
`train_pool['FlightDate'].isin(fold_dates)`.

**`gap=0`** (the default) is a deliberate choice, not an oversight: no feature in this
project is lagged or rolling, every predictor is known at booking time, so adjacent
dates share no constructed value that a zero gap could leak across a fold boundary.
Revisit this if a lagged or rolling feature is ever added.

**Seasonal limitation:** because the split is temporal, the test partition falls
entirely in Q4 (Oct-Dec). The Issue 10 holdout score therefore reflects winter
operations only, not a full-year average. `Month` and `DayOfWeek` stay in the feature
set for this reason, and this caveat should be repeated at Issues 6 and 15.


In [42]:
profile_cols = ['IATA_CODE_Reporting_Airline', 'Flight_Number_Reporting_Airline', 'Dest'] # Grouping by
for fold, train_fold, val_fold, train_dates, val_dates in make_cv_folds(train_pool, dates): # Calling the function to give the fold no, the rows of the train fold, the val fold, the dates that the train fold has, the val dates
    profile_median = train_fold.groupby(profile_cols)['ArrDelay'].median() # Grouping the rows in the train fold by the profile_cols and then finding the median for each group
    val_fold = val_fold.merge(profile_median.rename('pred').reset_index(), on=profile_cols, how='left') # This merges the profile_median, it first renames the ArrDelay column to pred to reduce confusion, then resets the index to turn the MultiIndex series into a small dataframe. It merges on the columns in profile_median, and if no match is found for that specific row then pred just returns NaN
    print(fold, val_fold['pred'].isna().sum(), len(val_fold)) # This basically talks about per fold, how many predictions had no match in the group and the len of val_fold is also there to see how much of the total rows does that number make

    # So 32% of the val rows in fold zero have no match at all amongst the groups that have been made, the plan is to not have them rely on per-profile median here because each group might only have 1-2 rows and the median from those rows is not really a good prediction since an outlier in such a small amount of rows can skew the median forward and therefore the predictions will be worse. The threshold is that each profile should have at least 10 rows for it to make a prediction using the per-profile grouping


    

0 16924 51554
1 9959 49684
2 6072 41431
3 11989 47184
4 5033 53895


fold 0 showed 32% of the val rows had no match at all in the training rows for that fold, but that number is actually lower than the real problem because right now the median is being computed for every profile that shows up even once, so profiles with just 1 or 2 rows are still getting a median instead of counting as "no match". the real number of rows that need a fallback is probably higher than 32% once a threshold gets added.

what the threshold does is for every row it just checks two things - is there a profile for it at all in the training data, and does that profile have enough rows (>=10) to trust the median. if both are true it uses the median from training, if not it falls back to the coarser grouping. thats it.


n=10 was picked by checking fold 0, the thinnest fold, since that's where the choice matters most. At n=10, only 5.0% of that fold's training rows sit in profiles too thin to use, even though 39.9% of individual profiles fall below it - most thin profiles just don't carry much row weight, so raising the bar to 10 doesn't throw away much data. n=5 barely changes that (1.2% excluded) but trusts a median computed from as few as 5-9 points, which is risky given ArrDelay's fat right tail (train pool p99=152, max=3359) - a single outlier flight could swing a median that small. Going higher costs data without buying stability: n=15 excludes 7.7% of rows and n=20 excludes 10.2%, so n=20 doubles the n=10 rate and n=15 is about one and a half times it. This threshold is applied the same way at every rung of the ladder and kept fixed across all 5 folds.

In [43]:
# Thresholding approach
mae_scores =[] # list to calculate MAE scores per fold
ladder_mae=[] #this is basically to calculate mae per fold of the rows that rung 1 was not able to predict because of not enough values in the profile or that there was no profile like that.They fell on to rung 2 or rung 3
flat_mae =[] # same as above but what this does is that it doesnt go through rung 2 it just replaces the prediciton for rows that rung 1 didnt predict with rung 3 which is the global median.This list stores the mae per fold in that case in order to compare it to the ladder mae
for fold, train_fold, val_fold, train_dates, val_dates in make_cv_folds(train_pool, dates):
    profile_stats = train_fold.groupby(profile_cols)['ArrDelay'].agg(['median', 'count'])
    reliable_profiles = profile_stats[profile_stats['count'] >= 10]['median'] # Gives the reliable profiles, the ones that have at least ten rows, and once they are identified their medians are taken. The code works by first giving a boolean mask for profiles with at least 10 rows, and then another filter on profile_stats keeps only those rows, after which their median is taken and count is discarded to keep only the reliable profiles
    val_fold = val_fold.merge(reliable_profiles.rename('pred').reset_index(), on=profile_cols, how='left') # Just does the merge with the validation fold, same as above
    val_fold['rung'] = np.where(val_fold['pred'].notna(),1,np.nan) # This basically creates a new column called rung to track which pred values are filled by which rung, since this is the first merge all the pred values come from rung 1, and this marks them as so in the rung column
    # print(fold, val_fold['pred'].isna().sum(), len(val_fold))

    # Fallback rate increased as predicted because the thin profiles are now not being considered, and if they are not being considered then there are more NaN's which increased the fallback rate

    coarse_stats = train_fold.groupby(['IATA_CODE_Reporting_Airline', 'DepHour'])['ArrDelay'].agg(['median', 'count'])
    reliable_coarse = coarse_stats[coarse_stats['count'] >= 10]['median'] # 10 is the count here as well, to ensure that only the profiles with at least 10 rows have the median sent as a prediction
    val_fold = val_fold.merge(reliable_coarse.rename('pred_coarse').reset_index(), on=['IATA_CODE_Reporting_Airline', 'DepHour'], how='left')
    unresolved = val_fold['pred'].isna() # Creates a snapshot of the pred column to find out rows which have the value of NaN
    val_fold['pred'] = val_fold['pred'].fillna(val_fold['pred_coarse']) # Replaces the prediction with the coarser prediction when pred is NaN, which means there was no match found in the most specific grouping, and replaces it with the prediction from a less specific grouping
    val_fold.loc[unresolved & val_fold['pred'].notna(),'rung'] = 2 # This line basically, what it does is, it finds the rows that were not resolved by rung 1 and does an elementwise AND with rows that are now resolved by both rung1 and rung2, and due to this elementwise AND, the only rows it shows true on are rows that were unresolved by the first rung which are now resolved by the second rung. The .loc[] then appropriately puts the value in the rung column as 2
    # print(fold, val_fold['pred'].isna().sum(), len(val_fold))

    still_unresolved = val_fold['pred'].isna() # Finds rows that are still unresolved after rung 1 and rung 2
    global_median = train_fold['ArrDelay'].median() # Calculate global median for rung 3
    val_fold['pred'] = val_fold['pred'].fillna(global_median) # Fill the NaN values with the global median, the first two rungs ensure that this is done for the least amount of rows as much as possible
    val_fold.loc[still_unresolved, 'rung'] = 3 # This labels the rows that were still unresolved after rung 1 and rung 2, and assigns them the value of 3.
    # print(fold,val_fold['pred'].isna().sum())
    non_rung_1 = val_fold[val_fold['rung'] != 1] #gets all the rows that were not predicited with rung 1
    fold_ladder_mae = (non_rung_1['ArrDelay'] - non_rung_1['pred']).abs().mean() # computes the mae for rows that were not predicted with rung 1 and rung 2
    ladder_mae.append(fold_ladder_mae)
    
    fold_flat_mae = (non_rung_1['ArrDelay']- global_median).abs().mean() #calculates mae on the rows that were not predicted with rung 1 but basically uses global median as the prediction for those rows and calculates the mae accordingly
    flat_mae.append(fold_flat_mae) 
    fold_mae = (val_fold['ArrDelay'] - val_fold['pred']).abs().mean() # calculating MAE for a single fold 
    mae_scores.append(fold_mae) # add ing the MAE for a particular fold to the mae_scores list
    print(fold, val_fold['rung'].value_counts().sort_index().to_dict()) # Gives the breakdown of how many values have been influenced by which rungs

print(np.mean(mae_scores), np.std(mae_scores)) #calculates mean to get the typical error of this approach and std tells how much those numbers differ amongst seperate folds.

#Mean MAE is 20 minutes for this approach and std is small at 1.2 which means theres not much variation in the performance as seasons/time passes

print(np.mean(ladder_mae),np.std(ladder_mae)) # prints out the mean across all 5 folds and the standard deviation to see performance across all five folds
print(np.mean(flat_mae),np.std(flat_mae))# same as above but in the case of when all the non-rung 1 fall straight to rung 3
print(np.mean(flat_mae) - np.mean(ladder_mae)) #tests to see if rung 2 is actually helping compared to just falling back to rung 3

0 {1.0: 30283, 2.0: 20641, 3.0: 630}
1 {1.0: 39274, 2.0: 10143, 3.0: 267}
2 {1.0: 34109, 2.0: 7308, 3.0: 14}
3 {1.0: 34643, 2.0: 12371, 3.0: 170}
4 {1.0: 39394, 2.0: 14356, 3.0: 145}
19.951910939426405 1.209756490038425
20.42569170229878 1.433145591603604
20.84019793146333 1.3210408121534298
0.41450622916454805


The first rung 2 grouped by carrier and destination and only got a lift of 0.149 minutes over just falling straight to the flat global median, basically nothing since both were scoring around 20.7-20.8 minutes MAE anyway.

Instead of guessing a better grouping, 11 different groupings got tested on the same 5 folds - carrier alone, Dest alone, departure hour alone, and combos of carrier/Dest/departure hour/month/day of week. Carrier + DepHour won clearly and roughly tripled the lift to 0.415 minutes. Two things came out of that search. Destination actually hurts - carrier alone (lift 0.210) beat carrier + Dest (0.149), and carrier + Dest + month dropped to basically -0.001, which is no better than the flat median at all, so adding destination isn't adding signal, it's just splitting groups into smaller, noisier ones. Departure hour on the other hand does carry a real effect Dest doesn't - the median ArrDelay by scheduled departure hour on the train pool shows 6am flights running a median of -10 minutes while 11am to 9pm flights sit near 0, so that's about a 9-10 minute swing across the day, which is bigger than any route level difference, probably because of delay propagation, where a delay earlier in an aircraft's day compounds through its later legs so a flight scheduled later inherits more of the day's accumulated slippage. DepHour is just a groupby key here, not a model feature, so the HHMM midnight wraparound problem doesn't apply, since group labels don't carry any ordering - hour 23 and hour 0 being far apart numerically doesn't matter here.

Also checked how good any median based rung could realistically get. Predicting the flat global median for every row gets 20.337 MAE, and a cheating oracle that uses each profile's true median, computed directly on the validation rows it's predicting (not something available in reality, just a ceiling check), gets 18.636. So the entire gap available to any median per group baseline is only about 1.70 minutes, because ArrDelay's spread around its own median is huge (mean absolute deviation ~20.4 minutes, p10=-22, p90=+36) compared to the difference between any two groups' centers. The current ladder is already close to that ceiling. Worth keeping in mind for Issue 6 - a model that can't beat this baseline by much isn't necessarily broken, since the naive per-profile median is already close to the best a median only approach can do.

One caveat though, the winning grouping was picked by scoring 11 candidates on the same 5 TimeSeriesSplit folds used everywhere else in this notebook, which is a mild form of selection on the CV folds. Defensible here because a stronger baseline makes the later "does the model actually beat the baseline" comparison harder, not easier, but this should be said plainly in Issue 6 instead of left implicit.


In [44]:
df['dep_minutes'] = (df['CRSDepTime'] // 100) * 60 + (df['CRSDepTime'] % 100) # adds a new column to the df that computes the scheduled departure time in the form of minutes since midnight.
df['arr_minutes'] = (df['CRSArrTime'] // 100) * 60 + (df['CRSArrTime'] % 100) #same thing as above but for scheduled arrival time
df[['CRSDepTime', 'dep_minutes', 'CRSArrTime', 'arr_minutes']].describe() #the minimum is one here because it literally means 12:01 AM as in 1.000 means that 0001 basically and as an integer thats just 1.0000

df['dep_sin'] = np.sin(2 * np.pi * df['dep_minutes'] / 1440) #calculates the angle and then puts it in the sin function to get the coordinate on the circle this is done so that the gap at midnight between 1439 and 1 is no longer there and is consistent with reality where these are only 2 minutes apart
df['dep_cos'] = np.cos( 2 * np.pi * df['dep_minutes'] / 1440) # same but puts it in a cos function this is done so that well if we only used sin then two distinct angles could have the same coordinates which is why we also use cos.

df['arr_sin'] = np.sin(2 * np.pi * df['arr_minutes'] / 1440) #does the same thing as above but for arrival times
df['arr_cos'] = np.cos(2 * np.pi * df['arr_minutes'] / 1440)

CRSDepTime and CRSArrTime are clock readings in HHMM format, not real quantities, so handing them to a linear model as raw integers is a problem. 10:59 to 11:00 is one minute of real time but a 1100 - 1059 = 41 unit jump in the raw integer. Converting to minutes since midnight with `(t // 100) * 60 + (t % 100)` fixes that: same step is now a clean +1. This is also why the describe() minimum came out as 1 and not 0 - 1 here literally means 00:01, 12:01am, and as an integer that's just 1.0000, not a weird edge case.

That conversion doesn't fix the wraparound at the day boundary though. 23:59 becomes 1439 and 00:01 becomes 1 - two minutes apart in real time, but 1439 - 1 = 1438 apart as numbers. Still basically as broken as the raw HHMM version for anything crossing midnight.

The fix is to calculate the angle for each time and put it through sin and cos to get a coordinate on a circle, so the gap at midnight between 1439 and 1 stops existing and matches reality where those two times are only 2 minutes apart. Checked directly: 23:59 gives (sin, cos) = (-0.0044, 1.0000) and 00:01 gives (0.0044, 1.0000) - 0.0087 apart in that space instead of 1438 apart in raw minutes, which is correctly close. Sin alone isn't enough because if we only used sin then two distinct angles could land on the same coordinate (e.g. 6am and 6pm), which is why cos is also needed - together they pin down one unique point on the circle.

Same thing applies to arrival time, same formula, just on arr_minutes instead of dep_minutes.

Month and DayOfWeek technically wrap around too (Dec/Jan, Sun/Mon) but at 12 and 7 levels one-hot just handles it for free with coefficients that are still easy to read, so periodic encoding is only worth it here for the 1,440-value time columns.


In [45]:
categorical_cols = ['IATA_CODE_Reporting_Airline', 'Dest', 'Month', 'DayOfWeek']
ohe = OneHotEncoder(drop='first')

drop="first" for the linear model. one hot encoding IATA_CODE_Reporting_Airline gives 11 dummy columns that always sum to 1 per row, and the intercept is really just a coefficient times a hidden column thats always 1 too, so the intercept column and the sum of the dummies are literally identical row for row. that means you can subtract any number from the intercept and add it to every dummy coefficient and predictions dont change at all, so theres infinitely many equally valid coefficient settings, not one, which makes a number like "carrier AS adds 5 minutes" meaningless since it couldve just as easily been 8 or -200.

drop="first" removes one dummy column, so that category becomes all zeros instead of having its own column. now it has nothing left to absorb a compensating shift, so shifting the intercept always changes its prediction with nothing to cancel it out.


In [46]:
open_cols = ['IATA_CODE_Reporting_Airline','Dest'] #columns for which the open encoder will be used
closed_cols = ['Month', 'DayOfWeek'] #columns for which the closed encoder will be used

ohe_open = OneHotEncoder(drop='first', handle_unknown='infrequent_if_exist',min_frequency=4)


In [47]:
ohe_closed = OneHotEncoder(drop='first', categories=[list(range(1, 13)), list(range(1, 8))]) 

handle_unknown for unseen categories. drop="first" already makes the dropped category all zeros, and the default handle_unknown="ignore" makes any unseen category all zeros too, so they become the identical vector and an unknown gets predicted as the reference silently. infrequent_if_exist routes unknowns to a dedicated infrequent column instead, but only if something actually fell below min_frequency during fit. tried min_frequency=2 and nothing qualified - the thinnest Dest in folds 0 and 1 has 3 rows - so no bucket got built and sklearn 1.7.2 silently fell back to all zeros exactly like ignore. min_frequency=4 clears it.

the real failure turned out to be Month, not Dest. the folds are temporal so fold 0 only trains on Jan-Apr and May-Aug arrive in validation as unknown, in 3 of 5 folds, and no min_frequency fixes that since months aren't rare, they're absent. so the split is by whether the category set is closed, not cardinality - Month and DayOfWeek get explicit categories= (1-12, 1-7) so unknown is impossible, Dest and carrier keep min_frequency + handle_unknown since you can't enumerate them ahead of time. two encoders in the ColumnTransformer. carrier never gets a bucket at any threshold worth setting - its rarest is 148 rows in fold 0, the thinnest case, and 744 across the whole train pool - and no unseen carrier appears in any fold, but a new airline at SEA is still possible in deployment, so that one's handled as input validation in issue 14 rather than distorted into the encoder here.

In [48]:
numeric_cols = ['DayofMonth', 'CRSElapsedTime', 'Distance', 'dep_sin', 'dep_cos', 'arr_sin', 'arr_cos']

Year, Quarter, FlightDate and Flight_Number_Reporting_Airline are all still sitting in df but none of them get handed to the ColumnTransformer, and each one is left out for a different reason.

Year only takes two values in this data, 2024 and 2025, and the app predicts 2026 onwards, so every prediction it ever makes is for a year the model has never seen. As a number it would extrapolate a trend off two points, one-hot it would just land in the infrequent bucket or all zeros. Either way it absorbs variance during training and contributes nothing valid at prediction time.

Quarter is a deterministic function of Month, Q1 is always months 1-3, so its dummy columns can be reconstructed exactly from Month's dummies. That's perfect collinearity, the same problem drop="first" exists to fix, except added back in on purpose. It carries nothing Month doesn't already carry.

FlightDate never recurs, 2024-03-15 isn't happening again, so there's nothing in it to generalize from. It stays in the dataframe because it's the Issue 2 split key and the Issue 3 grouping key, not because it's a feature.

Flight_Number_Reporting_Airline is a cardinality problem. One-hot on flight numbers alone would be 2,591 columns on the full dataset, or 2,383 on the train pool, against about 120 for all four categoricals that were kept. The ≈2,698 figure in PROJECT_ISSUES.md isn't flight numbers on their own, it's carrier plus flight number plus Dest added together. It's the profile key the Issue 3 baseline groups on and it's the user's input in Issue 14, but the model doesn't get it.

Keeping a column because it's useful for splitting, grouping or the app is a fine reason to keep a column, it just isn't a reason to feed it to a model.

In [49]:
preprocessor = ColumnTransformer([
    ('open', ohe_open, open_cols),
    ('closed', ohe_closed, closed_cols),
    ('num', 'passthrough', numeric_cols),
])

The four categoricals can't just be handed to an encoder along with everything else, because an encoder one-hot encodes every column you give it. Hand it the whole dataframe and it treats each distinct Distance value as its own category and hands back hundreds of junk columns - Distance has 100 distinct values and CRSElapsedTime has 364, so those two alone would turn into about 460 columns of nonsense. So something has to route specific columns to specific transformers, and that's what ColumnTransformer does. You build it out of (name, transformer, columns) triples, where the name is just a label you pick, the transformer is the thing doing the work, and the columns list is all that transformer ever sees. The numerics get the literal string "passthrough", which means copy them through untouched.

What decides where a column goes isn't whether it's a number. Month and DayOfWeek are numbers, integers 1-12 and 1-7, and they still go to a one-hot encoder. The real test is whether the number is a label or a quantity. Distance 2400 genuinely is twice 1200, so doubling it means something, but Month 12 isn't twice Month 6, December isn't twice June, the number is just a name. Labels get encoded, quantities pass through. Same distinction as 4a, where 1435 looked like a number but was really a clock reading.

Two encoders instead of one, split by whether the category set is closed rather than by cardinality like you'd expect. Month and DayOfWeek can be written out in advance, there is no 13th month, so they get an explicit categories= and unknown becomes impossible. Dest and carrier can't be enumerated ahead of time since an airline can add a route or start new service at SEA, so they keep min_frequency and handle_unknown. Cardinality isn't what breaks - Month has only 12 values and fails, carrier has 11 and doesn't.

Month is actually the column that fails, not Dest. The folds are temporal so fold 0 only trains on Jan-Apr, and May through August then turn up in its validation as categories the encoder never saw, which happens in 3 of the 5 folds, and no min_frequency value fixes it because those months aren't rare, they're absent. categories= fixes the encoding but not the information - May's column is all zeros across every one of fold 0's training rows, so its coefficient should come out 0 and it'll still predict like the dropped reference month. That needs checking against the actual fitted coef_ before being claimed. The deployed model doesn't have this problem at all since it fits on the whole train pool where every month has at least 12,398 rows, so categories= here is mostly CV hygiene.

What comes out is one matrix with the blocks side by side in the order the transformers were listed, and it's a numpy array rather than a dataframe, so the column names are gone. get_feature_names_out() hands them back in matching order, which is what lets you line names up against coef_ later. The width isn't identical in every fold, because the encoder learns its category list from whatever rows it's fitted on, and fold 0's 41,901 rows only contain 85 destinations while fold 4's 231,754 rows contain 93. That's correct, not a bug. If fold 0's encoder knew about all 93 it would mean destinations from 2025 had leaked backwards into a model that's only supposed to know early 2024, which is the same mistake as shuffling a temporal split.

Which is also why the whole thing has to sit inside a Pipeline. Fitting the preprocessor once on the entire train pool and then slicing folds out of the result would do exactly that leak. Inside a Pipeline, pipe.fit runs fit_transform on only that fold's training rows and pipe.predict runs transform on the validation rows, so validation gets encoded using categories learned from training alone, and that fit versus transform asymmetry is guaranteed instead of being something you have to remember for every transformer on every fold.

In [54]:
fold0_train = next(make_cv_folds(train_pool,dates))[1] #gives the training fold for fold 0
ohe_open.fit(fold0_train[open_cols]) #this basically fits the open columns and does the actual encoding
print(ohe_open.infrequent_categories_) # prints out which carriers and which destination are in the infrequent bucket
unseen = ohe_open.transform(pd.DataFrame({'IATA_CODE_Reporting_Airline': ['ZZ'], 'Dest': ['ZZZ']})).toarray() #converts the strings to columns basically like OneHotEncoding would do,this is to check where it goes as in do these strings have the value of is_infrequent = 1 or not.
print(unseen)
print(ohe_open.get_feature_names_out())

[None, array(['HDN'], dtype=object)]
[[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]]
['IATA_CODE_Reporting_Airline_AS' 'IATA_CODE_Reporting_Airline_B6'
 'IATA_CODE_Reporting_Airline_DL' 'IATA_CODE_Reporting_Airline_F9'
 'IATA_CODE_Reporting_Airline_HA' 'IATA_CODE_Reporting_Airline_MQ'
 'IATA_CODE_Reporting_Airline_NK' 'IATA_CODE_Reporting_Airline_OO'
 'IATA_CODE_Reporting_Airline_UA' 'IATA_CODE_Reporting_Airline_WN'
 'Dest_ALW' 'Dest_ANC' 'Dest_ATL' 'Dest_AUS' 'Dest_BLI' 'Dest_BNA'
 'Dest_BOI' 'Dest_BOS' 'Dest_BUR' 'Dest_BWI' 'Dest_BZN' 'Dest_CHS'
 'Dest_CLE' 'Dest_CLT' 'Dest_CMH' 'Dest_CVG' 'Dest_DAL' 'Dest_DCA'
 'Dest_DEN' 'Dest_DFW' 'Dest_DTW' 'Dest_EUG' 'Dest_EWR' 'Dest_FAI'
 'Dest_FAT' 'Dest_FCA' 'Dest_FLL' 'Dest_GEG' 'Dest_GTF' 'Dest_HLN'
 

/opt/anaconda3/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Fitting was the confusing part at first. Before .fit() the encoder is basically an empty shell - it has settings like drop="first" and min_frequency but zero knowledge, and it can't encode anything because it doesn't know how many columns to make or which carrier goes in which position. .fit() is where it scans the rows and writes down the distinct categories per column, their sorted order, and which of them fall below min_frequency, and stores all of that on the object. That's what categories_ and infrequent_categories_ are, with the trailing underscore being sklearn's convention for "only exists after fitting". Then .transform() uses that stored table as a lookup so the same category always lands in the same column position. The reason it's two steps rather than one is that it's what makes leakage preventable - you fit on training rows only, then transform validation with that same table.

Ran it on fold 0 because that's the thinnest fold at 41,901 training rows and it's the exact case where min_frequency=2 had failed. The rarest Dest there has 3 rows, while folds 2 through 4 each have a destination with exactly 1 row, so those folds would have built a bucket anyway and hidden the bug.

infrequent_categories_ came back as [None, array(['HDN'])]. None for carrier means no bucket got built, which was already enough to know an unseen carrier would come out all zeros before transforming anything at all. HDN is Hayden/Steamboat Springs and it's a winter-only route, all 9 of its rows sit in March. Same story for the other seasonal thin ones - FCA has 6 rows all in January, EGE has 39 and SUN has 32, both spread across December to March, all four of them ski destinations. Worth being careful about what actually gets pooled though, because those four aren't it. On the full train pool at min_frequency=4 the pooled set is exactly CID, BUF, MDT and GRB with one row each - Cedar Rapids, Buffalo, Harrisburg and Green Bay, one-off single flights rather than seasonal routes. EGE, SUN and FCA all sit above the threshold there and never get pooled at all. HDN only gets pooled in fold 0 because that fold's window stops in April and hasn't accumulated its 9 rows yet.

Dest's category count does climb from 85 in fold 0 to 93 in fold 4, but not because of winters. The new arrivals are SIT in fold 1, which is Sitka and runs year round, then BUF, CID and MDT in fold 2, which are single flights in October and November, then EGE, GRB and SUN in fold 3, where only EGE and SUN are the ski ones, and finally HOU in fold 4, which is a summer route running June through September. Mostly isolated one-off flights, not a seasonal pattern.

Transforming a row that's unseen in both columns gave back a 94 wide vector, 10 carrier columns and 84 Dest columns, with a single 1 sitting at Dest_infrequent_sklearn and nothing else anywhere. So Dest worked, ZZZ got routed into the bucket next to HDN and is distinguishable from the dropped reference. Carrier is all zeros. The carrier feature names only list 10 of the 11 and the missing one is AA, so an unseen carrier encodes identically to American Airlines - the exact collision 4c exists to prevent, still live, on purpose, because no encoder setting fixes it honestly and it gets handled as input validation in Issue 14 instead.

Two things worth remembering from this. The UserWarning says columns [0, 1] will be encoded as all zeros, but that's wrong about column 1, since Dest clearly got a 1 - the warning flags any column containing an unknown without checking whether that column had a bucket to absorb it, so read the output rather than trusting the message. And the check has to be done per block, because if you just ask whether the whole 94 wide vector is all zeros the answer is no, thanks to Dest's 1, and you'd wrongly conclude carrier was protected too.